# Load the three datasets and make the splits
1. Download HelpSteer2, PKU-SafeRLHF and UltraFeedback in the same format (`prompt`, `chosen`, `rejected`).
2. Split each one into train / val / test and save them to `data/`.

No GPU needed. **Step 2 only needs to run once.** After the files are pushed to GitHub, everyone uses the same splits.

In [ ]:
import os, sys, getpass

if os.path.exists("/content"):   
    if not os.path.exists("/content/mfr-dpo"):
        !git clone -q https://github.com/prabudhd2003/mfr-dpo.git /content/mfr-dpo
    !git -C /content/mfr-dpo pull -q
    REPO = "/content/mfr-dpo"
else:                            
    REPO = ".."

sys.path.insert(0, f"{REPO}/src")   
import importlib, mfr_data
importlib.reload(mfr_data)         

/Users/prabudhdkandpal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


<module 'mfr_data' from '/Users/prabudhdkandpal/Desktop/CSCI544/code/mfr-dpo/notebooks/../src/mfr_data.py'>

## 1. Load

In [2]:
# optional 
os.environ["HF_TOKEN"] = getpass.getpass("Paste your HF token: ")

In [3]:
datasets = {
    "helpful": mfr_data.load_helpsteer2(),
    "safe": mfr_data.load_pku_saferlhf(),
    "quality": mfr_data.load_ultrafeedback(),
}

README.md: 0.00B [00:00, ?B/s]

preference/preference.jsonl.gz:   0%|          | 0.00/15.3M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

data/Alpaca-7B/train.jsonl:   0%|          | 0.00/77.5M [00:00<?, ?B/s]

...

Generating train_sft split:   0%|          | 0/61135 [00:00<?, ? examples/s]

Generating test_prefs split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/61135 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [5]:
import pandas as pd

pd.DataFrame({
    name: {
        "pairs": len(df),
        "unique prompts": df["prompt"].nunique(),
        "avg prompt chars": int(df["prompt"].str.len().mean()),
        "avg chosen chars": int(df["chosen"].str.len().mean()),
        "avg rejected chars": int(df["rejected"].str.len().mean()),
    }
    for name, df in datasets.items()
})

,helpful,safe,quality
pairs,2765,10796,42182
unique prompts,2762,8555,42174
avg prompt chars,390,129,671
avg chosen chars,1463,455,1239
avg rejected chars,1348,528,1003


Do the chosen responses actually look better? Change `i` to see other examples.

In [6]:
i = 0
for name, df in datasets.items():
    row = df.iloc[i]
    print(f"=============== {name} ===============")
    print("PROMPT:  ", row["prompt"][:500])
    print("\nCHOSEN:  ", row["chosen"][:500])
    print("\nREJECTED:", row["rejected"][:500], "\n")

=============== helpful ===============
PROMPT:   Please make a list of independent Fertility coaching and consulting services in USA

CHOSEN:   Sure, here is a list of independent Fertility coaching and consulting services in USA:

1. Fertility Focus LLC
2. Fertility Journey Inc.
3. Fertility Road LLC
4. Fertility Wellness LLC
5. The Fertility Coach LLC
6. Fertility Consulting Services LLC
7. Fertility Health Services LLC
8. Fertility Support Services LLC
9. Fertility Advocates LLC
10. Fertility Resource Group LLC
11. Fertility Solutions LLC
12. Fertility Consulting LLC
13. Fertility Advocates and Solutions LLC
14. Fertility Advocates a

REJECTED: Sure, here are some independent Fertility coaching and consulting services in the USA:

1. Fertility Authority
2. Fertility Solutions
3. Fertility Success Coaching
4. Fertility Consulting Services
5. Fertility Coaching and Consulting
6. Fertility Coaching and Consulting Services
7. Fertility Coaching and Consulting Services
8. Fertility Coac

## 2. Split (run once)
Every dataset gets the same split sizes, so each training stage sees the same amount of data.
If one dataset runs short (HelpSteer2 is the smallest), `make_splits` shrinks **every** train split to the same size and prints why.


In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")   # only the tokenizer, not the model
SIZES = {"train": 2000, "val": 200, "test": 300}

splits = mfr_data.make_splits(datasets, tokenizer, SIZES, max_tokens=1024, seed=0)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

helpful    2765 pairs | -0 shared prompts | -3 extra pairs per prompt | -154 too long | 2608 left
safe      10796 pairs | -0 shared prompts | -2241 extra pairs per prompt | -1 too long | 8554 left
quality   42182 pairs | -431 shared prompts | -8 extra pairs per prompt | -2776 too long | 38967 left

helpful  train 2000, val 200, test 300
safe     train 2000, val 200, test 300
quality  train 2000, val 200, test 300


In [8]:
# token lengths in the train splits
pd.concat({name: parts["train"][["prompt_tokens", "chosen_tokens", "rejected_tokens"]].describe().round()
           for name, parts in splits.items()}, axis=1)

helpful                                        safe                \
      prompt_tokens chosen_tokens rejected_tokens prompt_tokens chosen_tokens   
count        2000.0        2000.0          2000.0        2000.0        2000.0   
mean          102.0         284.0           261.0          55.0          84.0   
std           108.0         194.0           188.0          13.0          54.0   
min            30.0           2.0             2.0          32.0           3.0   
25%            43.0         130.0           111.0          44.0          51.0   
50%            58.0         261.0           237.0          53.0          77.0   
75%           104.0         396.0           364.0          63.0         105.0   
max           755.0         964.0           970.0         167.0         823.0   

                            quality                                
      rejected_tokens prompt_tokens chosen_tokens rejected_tokens  
count          2000.0        2000.0        2000.0          2000.0  
mean             97.0         158.0         258.0           213.0  
std              47.0         152.0         226.0           202.0  
min               3.0          34.0           2.0             2.0  
25%              69.0          53.0          59.0            54.0  
50%              93.0         106.0         196.0           150.0  
75%             119.0         198.0         420.0           312.0  
max             458.0         983.0         968.0           894.0

In [9]:
DATA_DIR = f"{REPO}/data"
mfr_data.save_splits(splits, DATA_DIR)
print(sorted(os.listdir(DATA_DIR)))

['helpful_test.jsonl', 'helpful_train.jsonl', 'helpful_val.jsonl', 'quality_test.jsonl', 'quality_train.jsonl', 'quality_val.jsonl', 'safe_test.jsonl', 'safe_train.jsonl', 'safe_val.jsonl']


To load the data:
```python
splits = mfr_data.load_splits(f"{REPO}/data")
splits["helpful"]["train"]
```